In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

%load_ext autoreload
%autoreload 2

/home/turbotowerlnx/Documents/Master/TLH/TLH-CharoCharito-Alexa/notebooks
/home/turbotowerlnx/Documents/Master/TLH/TLH-CharoCharito-Alexa/app


In [2]:
import os
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [3]:
PATH_CARPETA = "../data/pdfs/"
DB_DIR = "../data/db_data"
MODEL_NAME = "llama3.1"
EMBEDDING_MODEL = "nomic-embed-text"

In [4]:
def cargar_y_indexar():
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

    if os.path.exists(DB_DIR):
        print("--- Cargando base de datos existente... ---")
        return Chroma(persist_directory=DB_DIR, embedding_function=embeddings)

    print(f"--- Escaneando PDFs en {PATH_CARPETA} ---")
    
    # DirectoryLoader busca todos los archivos .pdf y usa PyPDFLoader para cada uno
    loader = DirectoryLoader(
        PATH_CARPETA,
        glob="./*.pdf",
        loader_cls=PyPDFLoader
    )
    
    docs = loader.load()
    print(f"Documentos cargados: {len(docs)} páginas encontradas.")

    # Dividimos el texto en trozos
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = text_splitter.split_documents(docs)

    print(f"Creando base de datos vectorial con {len(splits)} fragmentos...")
    vectorstore = Chroma.from_documents(
        documents=splits, 
        embedding=embeddings, 
        persist_directory=DB_DIR
    )
    return vectorstore

In [5]:
vectorstore = cargar_y_indexar()
llm = OllamaLLM(model=MODEL_NAME)

--- Escaneando PDFs en ../data/pdfs/ ---
Documentos cargados: 276 páginas encontradas.
Creando base de datos vectorial con 282 fragmentos...


In [6]:
system_prompt = (
    "Eres un asistente de investigación. Responde basándote estrictamente en el contexto."
    "IMPORTANTE: Al final de tu respuesta, indica el nombre del archivo y la página de donde "
    "proviene la información (esta info está en los metadatos 'source' y 'page')."
    "\n\n"
    "Contexto: {context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    ])

In [7]:
chain = create_retrieval_chain(
    vectorstore.as_retriever(search_kwargs={"k": 10}), 
    create_stuff_documents_chain(llm, prompt)
)

In [8]:
query = input("\nPregunta sobre tus documentos: ")

print("Buscando en la biblioteca...")
res = chain.invoke({"input": query})
print(f"\nRespuesta:\n{res['answer']}")

Buscando en la biblioteca...

Respuesta:
Según el texto proporcionado, hay varias métricas que se utilizan para evaluar el resumen automático. Algunas de ellas requieren referencia (es decir, un resumen humano como punto de comparación) y otras no.

**Métricas que requieren referencia:**

* BLEU (Papineni et al., 2002): mide la precisión de n-gramas entre el resumen generado y el de referencia.
* METEOR (Lavie y Denkowski, 2009): mejora a BLEU al incluir sinónimos, derivaciones y lematización.
* CHRF (Popović, 2015): evalúa la similitud a nivel de caracteres (n-gramas de letras).
* ROUGE (Recall-Oriented Understudy for Gisting Evaluation) (Lin, 2004): métrica estándar que mide el recuerdo (recall), evaluando cuánta información del resumen humano ha sido capturada por la máquina.
* BERTScore (T. Zhang et al., 2020): calcula la similitud semántica mediante embeddings, superando la limitación de buscar coincidencias exactas de palabras.

**Métricas que no requieren referencia:**

* SummaQ